# NLP Lab Assignment — Experiment 5
## Title: Implement Shallow and Deep Parsing
**Case Study Domain:** Terms and Conditions Summarizer (Amazon & Alibaba Agreements)

### Objective:
To implement Part-of-Speech (POS) Tagging, Shallow Parsing (NP/VP Chunking for rights and obligations), Deep Dependency Parsing (Subject-Verb-Object extraction), and Context-Free Grammar (CFG) Parse Tree generation on contractual sentences.

### 1. Part-of-Speech (POS) Tagging
Assign grammatical categories to words in contractual obligations.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import pandas as pd

legal_sentence = "The customer must maintain strict confidentiality of their account password."
tokens = word_tokenize(legal_sentence)
pos_tags = pos_tag(tokens)

df_pos = pd.DataFrame(pos_tags, columns=["Word/Token", "POS Tag"])
print("=== POS Tagging of Legal Obligation Sentence ===")
display(df_pos)

### 2. Shallow Parsing & Chunking (Noun Phrase & Verb Phrase Extraction)
Extract rights, actors, and obligations using custom Regexp Chunk Grammars.

In [ ]:
# Define grammar for: Noun Phrase (Actors/Objects) and Verb Phrase (Actions/Obligations)
chunk_grammar = r"""
    NP: {<DT|PRP\$>?<JJ.*>*<NN.*>+}   # Noun phrase: e.g. 'The customer', 'strict confidentiality'
    VP: {<MD>?<VB.*>+}                 # Verb phrase / modal obligation: e.g. 'must maintain'
"""

chunk_parser = nltk.RegexpParser(chunk_grammar)
tree = chunk_parser.parse(pos_tags)

print("=== Chunked Syntactic Structure ===\n", tree)

print("\n--- Extracted Syntactic Phrases ---")
for subtree in tree.subtrees():
    if subtree.label() in ["NP", "VP"]:
        phrase = " ".join(word for word, tag in subtree.leaves())
        print(f"[{subtree.label()}] -> {phrase}")

### 3. Deep Dependency Parsing (Subject-Verb-Object / Actor-Action-Target)
Extract semantic relations (*Who is liable to do what*) using spaCy dependency graph.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

clause_text = "Amazon Services LLC disclaims all warranties under Section 12."
doc = nlp(clause_text)

print("=== Deep Dependency Parse Tree Attributes ===")
dep_data = []
for token in doc:
    dep_data.append({
        "Token": token.text,
        "Dependency": token.dep_,
        "Head Word": token.head.text,
        "POS": token.pos_,
        "Children": [child.text for child in token.children]
    })
display(pd.DataFrame(dep_data))

# Extract SVO (Subject-Verb-Object)
subject = [tok.text for tok in doc if "subj" in tok.dep_]
verb = [tok.text for tok in doc if tok.dep_ == "ROOT"]
obj = [tok.text for tok in doc if "obj" in tok.dep_]

print(f"\n--- Semantic SVO Extraction ---")
print(f"Actor (Subject): {subject}")
print(f"Action (Root Verb): {verb}")
print(f"Target (Object): {obj}")

### 4. Context-Free Grammar (CFG) Parse Tree Generation
Define a formal legal sentence CFG and generate hierarchical syntax parse trees.

In [ ]:
# Context-Free Grammar for Legal Clauses
legal_cfg = nltk.CFG.fromstring("""
    S -> NP VP
    NP -> Det N | Det Adj N | N
    VP -> Modal V NP | V NP | VP PP
    PP -> P NP
    
    Det -> 'the' | 'The' | 'all' | 'our'
    Adj -> 'strict' | 'valid'
    N -> 'customer' | 'Amazon' | 'confidentiality' | 'warranties' | 'discretion'
    Modal -> 'must' | 'shall' | 'may'
    V -> 'maintain' | 'disclaim' | 'accept'
    P -> 'in' | 'under'
""")

cfg_sentence = ["The", "customer", "must", "maintain", "confidentiality"]
parser = nltk.ChartParser(legal_cfg)

print("=== CFG Parse Tree Generated for Legal Sentence ===")
for tree in parser.parse(cfg_sentence):
    print(tree)
    print("\nTree Pretty Print Representation:")
    tree.pretty_print()